#### Baseline CNN
#### Activation: LeakyReLU  
#### Optimizer: SGD (lr=0.0001)  
#### Dataset: CIFAR-10

In [ ]:
%%time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import numpy as np

# Limit GPU memory to 25% since we are sharing with 4 people
torch.cuda.set_per_process_memory_fraction(0.25, device=0)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
%%time

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download and load training set
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2
)

# Download and load test set
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2
)

# CIFAR-10 class labels
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print('Training samples:', len(trainset))
print('Test samples:', len(testset))

In [ ]:
%%time

class CNN(nn.Module):
    def __init__(self, activation_fn=nn.LeakyReLU()):
        super(CNN, self).__init__()
        self.activation_fn = activation_fn

        # Convolutional layers
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          # 32x16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          # 64x8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2)                           # 128x4x4
        )

        # Fully connected layers
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            self.activation_fn,
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

# Instantiate model with LeakyReLU
model = CNN(activation_fn=nn.LeakyReLU()).to(device)
print(model)

In [ ]:
%%time

# Loss function
criterion = nn.CrossEntropyLoss()

# SGD optimizer with lr=0.0001
optimizer = optim.SGD(model.parameters(), lr=0.0001)

# Tensorboard writer
writer = SummaryWriter('runs/baseline_sgd_leakyrelu')

print('Optimizer: SGD | LR: 0.0001 | Activation: LeakyReLU')

In [ ]:
%%time

NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100. * correct / total

    # Log to Tensorboard
    writer.add_scalar('Loss/train', epoch_loss, epoch)
    writer.add_scalar('Accuracy/train', epoch_acc, epoch)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

writer.close()
print('Training complete!')

In [ ]:
%%time

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_accuracy = 100. * correct / total
print(f'Test Accuracy (Baseline - SGD + LeakyReLU): {test_accuracy:.2f}%')

In [ ]:
# Tensorboard
%load_ext tensorboard
%tensorboard --logdir runs